<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/W1_minichatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai faiss-cpu PyPDF2 tiktoken gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')

In [3]:
import io
import pickle
import numpy as np
import faiss
import gradio as gr
import tiktoken

from openai import OpenAI
from PyPDF2 import PdfReader
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 4
SIMILARITY_THRESHOLD = 0.35

# reads the uploaded file
def read_pdf(file_bytes):
    text = []
    pdf = PdfReader(io.BytesIO(file_bytes))
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

def read_text_file(file_bytes):
    try:
        return file_bytes.decode("utf-8")
    except UnicodeDecodeError:
        return file_bytes.decode("latin-1", errors="ignore")

def load_documents(uploaded_files):
    docs = []
    for filename, file_bytes in uploaded_files.items():
        lower = filename.lower()

        try:
            if lower.endswith(".pdf"):
                text = read_pdf(file_bytes)
            elif lower.endswith(".txt") or lower.endswith(".md") or lower.endswith(".markdown"):
                text = read_text_file(file_bytes)
            else:
                print(f"Skipping unsupported file: {filename}")
                continue

            if text and text.strip():
                docs.append({
                    "source": filename,
                    "text": text
                })
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    return docs

#chuncks of documents
encoding = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    tokens = encoding.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_str = encoding.decode(chunk_tokens).strip()

        if chunk_str:
            chunks.append(chunk_str)

        start += (chunk_size - overlap)

    return chunks

def build_chunks(docs):
    chunked_docs = []

    for doc in docs:
        parts = chunk_text(doc["text"])
        for i, part in enumerate(parts):
            chunked_docs.append({
                "source": doc["source"],
                "chunk_id": i,
                "text": part
            })

    return chunked_docs

#chunks embeddings
def get_embeddings(text_list, model=EMBED_MODEL, batch_size=64):
    all_embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]

        response = client.embeddings.create(
            model=model,
            input=batch
        )

        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)

    return all_embeddings

#searches for most neigbour similar
class FAISSVectorStore:
    def __init__(self):
        self.index = None
        self.metadata = []

    def build(self, chunked_docs):
        texts = [item["text"] for item in chunked_docs]
        embeddings = get_embeddings(texts)

        matrix = np.array(embeddings).astype("float32")
        faiss.normalize_L2(matrix)

        dimension = matrix.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(matrix)

        self.metadata = chunked_docs

    def search(self, query, top_k=TOP_K):
        query_embedding = client.embeddings.create(
            model=EMBED_MODEL,
            input=[query]
        ).data[0].embedding

        query_vector = np.array([query_embedding]).astype("float32")
        faiss.normalize_L2(query_vector)

        scores, indices = self.index.search(query_vector, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1:
                item = self.metadata[idx].copy()
                item["score"] = float(score)
                results.append(item)

        return results

    def save(self, index_path="faiss.index", meta_path="faiss_meta.pkl"):
        faiss.write_index(self.index, index_path)
        with open(meta_path, "wb") as f:
            pickle.dump(self.metadata, f)

    def load(self, index_path="faiss.index", meta_path="faiss_meta.pkl"):
        self.index = faiss.read_index(index_path)
        with open(meta_path, "rb") as f:
            self.metadata = pickle.load(f)

    #loads doc
docs = load_documents(uploaded)
print(f"Loaded {len(docs)} documents.")

chunked_docs = build_chunks(docs)
print(f"Created {len(chunked_docs)} chunks.")

vector_store = FAISSVectorStore()
vector_store.build(chunked_docs)
vector_store.save()

print("Knowledge base built successfully!")

#Answering through llm
def build_context(retrieved_chunks):
    context_parts = []

    for item in retrieved_chunks:
        block = (
            f"Source: {item['source']} | Chunk: {item['chunk_id']} | Score: {item['score']:.4f}\n"
            f"{item['text']}"
        )
        context_parts.append(block)

    return "\n\n---\n\n".join(context_parts)

def is_question_related(retrieved_chunks, threshold=SIMILARITY_THRESHOLD):
    if not retrieved_chunks:
        return False

    best_score = retrieved_chunks[0]["score"]
    return best_score >= threshold

def answer_question(query):
    results = vector_store.search(query)
    related = results and results[0]["score"] >= SIMILARITY_THRESHOLD

    if related:
        prompt = f"""Answer using the document context below.

Context:
{build_context(results)}

Question:
{query}
"""
        system = "You are a helpful Knowledge Assistant. Use the provided context as the main source of truth."
        mode = "Document-based answer"
        temperature = 0.2
    else:
        prompt = query
        system = "You are a helpful AI assistant. The uploaded documents are not strongly relevant, so answer using general knowledge."
        mode = "General LLM answer"
        temperature = 0.5

    res = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ]
    )

    answer = res.choices[0].message.content
    sources = "\n".join(
        f"- {r['source']} (chunk {r['chunk_id']}, score={r['score']:.3f})" for r in results
    ) if results else "No document chunks retrieved."

    return f"Mode: {mode}\n\n{answer}\n"


with gr.Blocks() as demo:
    gr.Markdown("# Mini ChatGPT Knowledge Assistant")
    gr.ChatInterface(
        fn=lambda message, history: answer_question(message),
        textbox=gr.Textbox(
            label="Ask a Question",
            placeholder="Ask about your document or anything else..."
        )
    )

demo.launch(share=True)


Saving Genai.txt to Genai.txt
Uploaded files: ['Genai.txt']
Loaded 1 documents.
Created 4 chunks.
Knowledge base built successfully!


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ab86492e56d69dce7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
